In [ ]:
!pip install torch scikit-learn pandas joblib

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import joblib

In [ ]:
df = pd.read_csv("/content/mindlens_history.csv")

In [ ]:
df.head()

,user_id,day,depression,anxiety,stress,risk
0,0,0,0.3324,0.3304,0.3670,0.3165
1,0,1,0.3061,0.3038,0.2862,0.3324
2,0,2,0.3190,0.3123,0.1952,0.2714
3,0,3,0.2999,0.3018,0.2300,0.3370
4,0,4,0.3608,0.4589,0.3506,0.3309


In [ ]:
df.shape

(600000, 6)

In [ ]:
features = ['depression','anxiety','stress','risk']

scaler = MinMaxScaler()

df[features] = scaler.fit_transform(df[features])

joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [ ]:
SEQ_LEN = 7

X = []
y = []

for user_id in df['user_id'].unique():

    user_df = df[df['user_id'] == user_id].sort_values("day")

    values = user_df[features].values

    for i in range(len(values) - SEQ_LEN):
        X.append(values[i:i+SEQ_LEN])
        y.append(values[i+SEQ_LEN][3])   # next risk

In [ ]:
X = np.array(X)
y = np.array(y)

In [ ]:
print(X.shape, y.shape)

(530000, 7, 4) (530000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X).float()
        self.y = torch.tensor(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(
    EmotionDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    EmotionDataset(X_test, y_test),
    batch_size=64
)

In [ ]:
class EmotionLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(64,1)

    def forward(self,x):
        out,_ = self.lstm(x)
        out = out[:,-1,:]
        out = self.fc(out)
        return out.squeeze()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(device)

cuda


In [ ]:
model = EmotionLSTM().to(device)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(X_batch)

        loss = criterion(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 36.9193
Epoch 2, Loss: 33.5213
Epoch 3, Loss: 30.7713
Epoch 4, Loss: 29.9349
Epoch 5, Loss: 29.5644
Epoch 6, Loss: 29.2503
Epoch 7, Loss: 29.0933
Epoch 8, Loss: 28.9423
Epoch 9, Loss: 28.8085
Epoch 10, Loss: 28.7613


In [ ]:
torch.save(model.state_dict(), "emotion_lstm.pth")
print("Saved successfully")

Saved successfully


In [ ]:
from google.colab import files

files.download("emotion_lstm.pth")
files.download("scaler.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>